# vEMDestripe inference and comparison

Run the Bumbarger13 demonstration, inspect XY, XZ and YZ views, and
compare the saved outputs of vEMDestripe, EMISAC, FECNet and SSCOR.
The demonstration covers sections 777–808 (32 × 1024 × 1024).

**Workflow** — network inference → inference postprocessing → save outputs → evaluate.


## 1. Connect and run

Select **Runtime → Run all**. The default **published_full_volume** mode
uses verified saved network outputs and works on a CPU runtime.
Optional **live_demo** performs network inference on the 32 supplied sections
with a GPU, then runs inference postprocessing. Use a fresh runtime when
switching modes.


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, time, zipfile

GITHUB_OWNER, REPOSITORY = "simple-happy-802", "vEMDestripe"
REPOSITORY_REF = "bumbarger-demo-metrics-v2"
RELEASE_TAG = "demo-reference-v3"  # immutable input, network-output and baseline assets
ASSET_NAME = "vemdestripe_bumbarger_reference_pipeline_v3.zip"
ASSET_SHA256 = "9aa8bdbf82dbbafb4473df73abdcc2dea6a9853c4c6aa135cb4a8809f61ff669"
ASSET_URL = (f"https://github.com/{GITHUB_OWNER}/{REPOSITORY}/releases/"
             f"download/{RELEASE_TAG}/{ASSET_NAME}")
VEM_RUN_MODE = 'published_full_volume'  # or 'live_demo' (GPU, separate 32-section context)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RUNTIME_ROOT = Path('/content') if IN_COLAB else Path.cwd()/'original793_demo_runtime'
PROJECT_ROOT = RUNTIME_ROOT/f'{REPOSITORY}-{REPOSITORY_REF}'
DOWNLOAD_ROOT, RESULT_ROOT = RUNTIME_ROOT/'downloads', RUNTIME_ROOT/'original793_comparison_results'
for folder in (RUNTIME_ROOT, DOWNLOAD_ROOT, RESULT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

def contains_cjk(text):
    return re.search(r'[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff\U00020000-\U000323af]', text) is not None

def run_command(command, **kwargs):
    result = subprocess.run([str(x) for x in command], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    if result.returncode:
        lines = [line for line in result.stdout[-12000:].splitlines() if not contains_cjk(line)]
        if lines:
            print('\n'.join(lines))
        raise RuntimeError(f'Command failed (exit {result.returncode})')
    return result.stdout
print(f'Runtime: {RUNTIME_ROOT}; Colab: {IN_COLAB}')

In [ ]:
repo_url = f'https://github.com/{GITHUB_OWNER}/{REPOSITORY}.git'
if not (PROJECT_ROOT/'.git').is_dir():
    run_command(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
                 repo_url, PROJECT_ROOT])
commit = run_command(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD']).strip()
tagged_commit = run_command(['git', '-C', PROJECT_ROOT, 'rev-parse',
                             REPOSITORY_REF+'^{commit}']).strip()
if commit != tagged_commit:
    raise RuntimeError('Existing checkout is not at the versioned demo tag. Use a fresh runtime.')
if run_command(['git', '-C', PROJECT_ROOT, 'status', '--porcelain',
                '--untracked-files=no']).strip():
    raise RuntimeError('The demo source checkout has modified tracked files; use a fresh runtime.')
print('Installing demo dependencies; the existing PyTorch build is retained if compatible.')
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-e',
             f'{PROJECT_ROOT}[evaluation]', 'pandas>=2.2,<3',
             'ipywidgets>=8,<9', 'requests>=2.32,<3', 'h5py>=3.11,<4', 'numba>=0.62,<1'])
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT/'colab'))
print(f'Versioned project source: {REPOSITORY_REF}, commit {commit}')

## 2. Download and verify assets

The versioned v3 bundle supplies the original input, saved network outputs,
source histograms, guidance references and baseline results. SHA256 checks
verify the downloaded archive and its files. The pipeline saves the current
vEMDestripe final outputs before evaluation.


In [ ]:
import requests
from reference_pipeline_demo import sha256_file, safe_extract_bundle, verify_bundle

archive = DOWNLOAD_ROOT/ASSET_NAME
if not archive.is_file():
    partial = archive.with_suffix('.zip.part')
    print('Downloading the original-source demo bundle…')
    with requests.get(ASSET_URL, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with partial.open('wb') as stream:
            for chunk in response.iter_content(1024*1024):
                if chunk:
                    stream.write(chunk)
    if sha256_file(partial) != ASSET_SHA256:
        raise RuntimeError('Downloaded archive checksum mismatch; do not use these results.')
    partial.replace(archive)
if sha256_file(archive) != ASSET_SHA256:
    raise RuntimeError('Cached archive checksum mismatch. Start with a fresh runtime.')
ASSET_ROOT = safe_extract_bundle(archive, DOWNLOAD_ROOT/'official_bundle', ASSET_SHA256)
manifest = verify_bundle(ASSET_ROOT)
print(f'Verified archive: {archive.stat().st_size/1024**2:.1f} MiB')
print('Every bundled file passed its SHA256 check.')

## 3. Original-source EMISAC

The comparison includes verified outputs from the official EMISAC source
and its legacy environment. The bundle retains its original H5, source,
parameters, dependency versions and logs, with reproduction instructions.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, HTML, Code, FileLink
from official_demo import official_metadata, verify_emisac_pixels

emisac_metadata = official_metadata(ASSET_ROOT)
emisac_pixel_check = verify_emisac_pixels(ASSET_ROOT)
provenance_root = ASSET_ROOT/'provenance'/'emisac_official'
display(Markdown(
    '**EMISAC source and output verification passed.**'))
display(Markdown(f'[Official source and reproducibility instructions]('
                 f'https://github.com/{GITHUB_OWNER}/{REPOSITORY}/blob/{commit}/'
                 'colab/official_emisac/REPRODUCE.md)'))

## 4. vEMDestripe inference

This notebook uses the released checkpoint with mean reference fusion and
16×16 block averaging. The default mode loads saved network outputs verified
against the complete 2651-section source volume. Live mode runs the checkpoint
on the 32 supplied sections. Inference postprocessing follows in the next step.


In [ ]:
import torch
checkpoint = PROJECT_ROOT/'checkpoints/vemdestripe_seed42/best.pth'
CHECKPOINT_SHA256 = '9a01cdef1f09716fe87369b057633f9e1278e86e77cbc70eb66b75798709c44a'
if sha256_file(checkpoint) != CHECKPOINT_SHA256:
    raise RuntimeError('Released checkpoint checksum mismatch')
if VEM_RUN_MODE == 'published_full_volume':
    NETWORK_RAW_DIR = ASSET_ROOT/'precomputed/vEMDestripe'
    VEM_SOURCE = 'saved full-volume network output (2651 sections)'
elif VEM_RUN_MODE == 'live_demo':
    if not torch.cuda.is_available():
        raise RuntimeError('live_demo needs a GPU. Select a GPU or use published_full_volume.')
    NETWORK_RAW_DIR = RESULT_ROOT/'vEMDestripe_network_raw_live'
    NETWORK_RAW_DIR.mkdir(parents=True,exist_ok=True)
    command = [sys.executable, PROJECT_ROOT/'infer.py',
        '--input_dir', ASSET_ROOT/'input', '--output_dir', NETWORK_RAW_DIR,
        '--model', checkpoint, '--reference', ASSET_ROOT/'reference',
        '--num_reference','16','--reference_mode','fused',
        '--reference_preprocess','block_mean','--struct_block_size','16',
        '--reference_selection','uniform','--reference_seed','42',
        '--image_size','1024','--inference_size','1024',
        '--no_sliding_window','--no_z_context','--no_z_postprocess',
        '--no_preview','--disable_progress_bar','--no_photometric_calibration']
    started=time.time(); log=run_command(command)
    (RESULT_ROOT/'network_inference.log').write_text(log,encoding='utf-8')
    VEM_SOURCE = 'live network output (32 sections)'
    print(f'Live network inference: {(time.time()-started)/60:.1f} min')
else:
    raise ValueError('VEM_RUN_MODE must be published_full_volume or live_demo')
print('vEMDestripe network output ready:', VEM_RUN_MODE)

## 5. Inference postprocessing

Prepare the final PNGs, TIFF and processing receipt for inspection and evaluation.
The download includes the processing record.


In [ ]:
from original_target_demo import materialize_original_target_pipeline, verify_exported_raw_files, TARGET_SECTION_ID
TARGET_PATH=ASSET_ROOT/'input'/f'slice_{TARGET_SECTION_ID:05d}.png'
FINAL_VEM_DIR=RESULT_ROOT/('vEMDestripe_original793_final_'+VEM_RUN_MODE)
marker=FINAL_VEM_DIR/'CALIBRATION.json'
if marker.is_file():
    calibration=json.loads(marker.read_text(encoding='utf-8'))
    expected_count=2651 if VEM_RUN_MODE=='published_full_volume' else 32
    if (calibration.get('target_section_id')!=793
        or calibration.get('target_source')!='single_original_slice'
        or calibration['target_file']['sha256']!=sha256_file(TARGET_PATH)
        or calibration['source_statistics']['section_count']!=expected_count):
        raise RuntimeError('Saved processing settings differ. Use a fresh runtime.')
    verify_exported_raw_files(calibration,NETWORK_RAW_DIR)
    for item in [*calibration['output_files'],calibration['output_stack']]:
        if sha256_file(FINAL_VEM_DIR/item['name'])!=item['sha256']:
            raise RuntimeError('Saved final output changed; do not evaluate it.')
else:
    calibration=materialize_original_target_pipeline(ASSET_ROOT,FINAL_VEM_DIR,
        source_dir=NETWORK_RAW_DIR if VEM_RUN_MODE=='live_demo' else None)
print('Inference postprocessing complete. Saved final outputs and processing receipt.')


## 6. Inspect outputs

**Final pipeline outputs** shows the saved results used for evaluation.
**Network-only diagnostic** shows the vEMDestripe intermediate for inspection.
All panels share the 0–255 display scale. XY uses its native grid. With a
voxel spacing of 3.9 × 3.9 × 50 nm, XZ and YZ are linearly resliced along Z
at the physical spacing ratio (50/3.9 ≈ 12.82×).
The 32 acquired layers give 410 displayed rows. The brightness profile
uses the 32 original layer positions.

In [ ]:
from evaluation.demo import load_saved_methods
from reference_pipeline_demo import method_directories
from demo_metrics import METHOD_ORDER, load_png_volume, plot_xy, plot_z_profiles
import ipywidgets as widgets
from ipywidgets import interact,IntSlider
FINAL_METHOD_DIRS=method_directories(ASSET_ROOT,FINAL_VEM_DIR)
volumes,section_ids=load_saved_methods(FINAL_METHOD_DIRS,expected_ids=range(777,809))
network_volume,network_ids=load_png_volume(NETWORK_RAW_DIR)
if network_ids!=section_ids or network_volume.shape!=volumes['vEMDestripe'].shape:
    raise RuntimeError('Network and final display windows differ')
network_diagnostic={**volumes,'vEMDestripe':network_volume}
sources={'Original':'delivered input, unchanged',
         'EMISAC':'unchanged official source + legacy environment, precomputed; no new calibration',
         'FECNet':'delivered grayscale adaptation/fine-tuned output; no new calibration',
         'SSCOR':'delivered dataset-specific seed-42 output; no new calibration',
         'vEMDestripe':VEM_SOURCE}
print('Loaded comparison outputs:', ', '.join(METHOD_ORDER))
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
display_mode=widgets.ToggleButtons(options=['Final pipeline outputs','Network-only diagnostic'],
                                   value='Final pipeline outputs',description='Display:')
display(display_mode)
@interact(z=IntSlider(min=0,max=31,value=16,description='Z offset'),mode=display_mode)
def inspect_xy(z,mode):
    print(mode)
    plot_xy(volumes if mode=='Final pipeline outputs' else network_diagnostic,section_ids,z)


In [ ]:
from bumbarger_evaluation import plot_demo_side

@interact(y=IntSlider(min=32,max=991,step=8,value=512,description='Y'),mode=display_mode)
def inspect_xz(y, mode):
    print(mode)
    plot_demo_side(volumes if mode == 'Final pipeline outputs' else network_diagnostic, 'xz', y)

@interact(x=IntSlider(min=32,max=991,step=8,value=512,description='X'),mode=display_mode)
def inspect_yz(x, mode):
    print(mode)
    plot_demo_side(volumes if mode == 'Final pipeline outputs' else network_diagnostic, 'yz', x)

@interact(mode=display_mode)
def inspect_brightness(mode):
    print(mode)
    plot_z_profiles(volumes if mode == 'Final pipeline outputs' else network_diagnostic, section_ids)

## 7. Evaluate saved outputs

The table reports **clean-kidney-reference NIQE** for XY, XZ and YZ (lower
means closer to the reference statistics) and **ASID** for axial brightness
variation on the native 32 layers (lower means more stable layer means).

Each NIQE entry averages the same fixed positions across methods: eight native
XY planes and six physical XZ or YZ planes per axis. Side planes share the
display's 12.82× Z reslicing and use the lateral crop [32,992). The export
includes each score, coordinate and valid 96-pixel patch count.

ASID is the median absolute deviation of signed differences between adjacent
whole-layer mean intensities. Evaluation reads the saved uint8 images using
their existing intensity scale. Bumbarger13 has no paired clean target.
See the [evaluation protocol](https://github.com/simple-happy-802/vEMDestripe/blob/bumbarger-demo-metrics-v2/colab/NIQE_PROTOCOL.md).

In [ ]:
from bumbarger_evaluation import evaluate_demo

saved_volumes, saved_ids = load_saved_methods(FINAL_METHOD_DIRS, expected_ids=range(777, 809))
metrics, niqe_sections = evaluate_demo(saved_volumes, saved_ids)
del saved_volumes
metrics.to_csv(RESULT_ROOT/'metrics_final_outputs.csv')
niqe_sections.to_csv(RESULT_ROOT/'metrics_niqe_sections.csv', index=False)
display(metrics.rename(columns={'NIQE_XY': 'NIQE XY ↓', 'NIQE_XZ': 'NIQE XZ ↓',
                                'NIQE_YZ': 'NIQE YZ ↓', 'ASID': 'Native-Z ASID ↓'}).round(4))
print(f"NIQE uses frozen clean-kidney statistics. "
      f"Valid patches per scored plane: {niqe_sections.patch_count.min()}–{niqe_sections.patch_count.max()}.")

## 8. Source code

Browse the network, inference postprocessing, export and evaluation code.
The source viewer also links the pinned EMISAC, FECNet and SSCOR implementations
and their processing provenance.


In [ ]:
from functools import lru_cache
source_specs = {
    'vEMDestripe': {'repo': f'{GITHUB_OWNER}/{REPOSITORY}', 'commit': commit,
        'paths': ['vemdestripe/model.py', 'infer.py', 'vemdestripe/calibration.py', 'calibrate_volume.py', 'colab/original_target_demo.py', 'evaluation/demo.py', 'colab/bumbarger_evaluation.py', 'colab/NIQE_PROTOCOL.md']},
    'EMISAC': {'repo': 'azadis/EMISAC',
        'commit': '7a82a17b7bbb4ca14dff2d1c1a4a018fc8cebaf5',
        'paths': ['EM_normalization.py', 'normalize_partition.py']},
    'FECNet': {'repo': 'KevinJ-Huang/FECNet',
        'commit': 'fc5a9eb224c448d190919558cc87ea7b7447f7f4',
        'paths': ['FastFourierSICE/models/archs/EnhanceN_arch.py', 'FastFourierSICE/test.py']},
    'SSCOR': {'repo': 'AI-XPM-group/SSCOR',
        'commit': '985479cd79bcf1359e3d9ba44bacd5f372eb2e60',
        'paths': ['models/networks.py', 'restore.py', 'train.py']}}

@lru_cache(maxsize=20)
def fetch_source(method, file):
    if method == 'vEMDestripe':
        return (PROJECT_ROOT/file).read_text(encoding='utf-8')
    if method == 'EMISAC':
        return (provenance_root/'source'/file).read_text(encoding='utf-8')
    spec = source_specs[method]
    url = f"https://raw.githubusercontent.com/{spec['repo']}/{spec['commit']}/{file}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text

method_choice = widgets.Dropdown(options=list(source_specs), description='Method:')
path_choice = widgets.Dropdown(options=source_specs['vEMDestripe']['paths'],
                              description='File:', layout=widgets.Layout(width='650px'))
show_all = widgets.Checkbox(value=False, description='Show complete file')
source_output = widgets.Output()
def render_source(change=None):
    with source_output:
        source_output.clear_output(wait=True)
        method, file = method_choice.value, path_choice.value
        if file not in source_specs[method]['paths']:
            return
        spec = source_specs[method]
        url = f"https://github.com/{spec['repo']}/blob/{spec['commit']}/{file}"
        display(Markdown(f'**{method}** — [complete pinned source]({url})'))
        try:
            lines = fetch_source(method, file).splitlines()
            selected = lines if show_all.value else lines[:200]
            preview = chr(10).join(selected)
            if contains_cjk(preview):
                print('Use the pinned source link above to read this file in its original language.')
                return
            display(Code(preview, language='python'))
            print(f'Showing {len(selected)}/{len(lines)} lines. Result source: {sources[method]}')
        except Exception:
            print('Inline source unavailable. Use the pinned source link above.')
def change_method(change):
    path_choice.options = source_specs[change['new']]['paths']
    render_source()
method_choice.observe(change_method, names='value')
path_choice.observe(render_source, names='value')
show_all.observe(render_source, names='value')
display(widgets.VBox([method_choice, path_choice, show_all, source_output]))
render_source()

## 9. Export

Download final outputs, network intermediates, processing receipts and metrics.
The ZIP also includes EMISAC source and logs, data attribution, the bundle
manifest and the integrity checks used by this run.


In [ ]:
verify_bundle(ASSET_ROOT)
from bumbarger_evaluation import PRIOR_SHA256, REFERENCE_DOI, SPACING_XYZ_NM, Z_RATIOS
runtime_provenance={'project_commit':commit,'version_tag':REPOSITORY_REF,
    'asset_sha256':ASSET_SHA256,'local_z_half_open':[777,809],
    'vEMDestripe_mode':VEM_RUN_MODE,'sources':sources,
    'calibrated_methods':['vEMDestripe'],'evaluation_normalization':False,
    'evaluation_input':'saved final PNG files','no_paired_clean_target':True,
    'evaluation_metrics':['NIQE_XY','NIQE_XZ','NIQE_YZ','ASID'],
    'niqe_reference':'axis-specific clean kidney', 'niqe_reference_doi':REFERENCE_DOI,
    'niqe_reference_sha256':PRIOR_SHA256, 'voxel_spacing_xyz_nm':SPACING_XYZ_NM,
    'niqe_z_ratio_by_axis':Z_RATIOS,'niqe_side_lateral_bounds':[32,992],
    'native_z_asid':True,
    'calibration_source_sections':calibration['source_statistics']['section_count'],
    'calibration_target':'single Original slice_00793.png',
    'target_section_id':793,'target_sha256':sha256_file(TARGET_PATH),
    'demo_target_override':True,'production_reference_default_changed':False}
(RESULT_ROOT/'runtime_provenance.json').write_text(json.dumps(runtime_provenance,indent=2),encoding='utf-8')
for method,folder in FINAL_METHOD_DIRS.items():
    destination=RESULT_ROOT/'final_outputs'/method
    destination.mkdir(parents=True,exist_ok=True)
    for path in Path(folder).glob('slice_*.png'):
        shutil.copy2(path,destination/path.name)
shutil.copytree(NETWORK_RAW_DIR,RESULT_ROOT/'network_only_vEMDestripe',dirs_exist_ok=True)
shutil.copytree(ASSET_ROOT/'calibration',RESULT_ROOT/'historical_v3_source_provenance',dirs_exist_ok=True)
shutil.copytree(provenance_root,RESULT_ROOT/'emisac_official_provenance',dirs_exist_ok=True)
for source in (ASSET_ROOT/'DATA_LICENSE.txt',PROJECT_ROOT/'colab/DATA_LICENSE.md',
               PROJECT_ROOT/'colab/THIRD_PARTY_METHODS.md', PROJECT_ROOT/'colab/NIQE_PROTOCOL.md',
               PROJECT_ROOT/'colab/clean_kidney_niqe_prior.npz'):
    if source.is_file():shutil.copy2(source,RESULT_ROOT/source.name)
shutil.copy2(ASSET_ROOT/'manifest.json',RESULT_ROOT/'bundle_manifest.json')
shutil.copy2(TARGET_PATH,RESULT_ROOT/'calibration_target_Original_slice_00793.png')
export_zip=Path(shutil.make_archive(str(RUNTIME_ROOT/'vemdestripe_original793_demo_results'),'zip',RESULT_ROOT))
print(f'Exported {export_zip.name} ({export_zip.stat().st_size/1024**2:.1f} MiB)')
if IN_COLAB:
    from google.colab import files
    button=widgets.Button(description='Download results ZIP',icon='download')
    button.on_click(lambda _:files.download(str(export_zip)))
    display(button)
else:display(FileLink(str(export_zip)))